This will load the imports

In [1]:
!git clone https://github.com/karpathy/nanogpt

!pip install tokenizers

Cloning into 'nanogpt'...
remote: Enumerating objects: 689, done.
remote: Total 689 (delta 0), reused 0 (delta 0), pack-reused 689 (from 1)
Receiving objects: 100% (689/689), 981.26 KiB | 3.20 MiB/s, done.
Resolving deltas: 100% (380/380), done.


This will load the model (to run it yourself make sure to change the location)

In [2]:
from google.colab import files
from google.colab import drive
drive.mount('/content/drive')



import sys
sys.path.append('/content/nanogpt')

import torch
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.decoders import ByteLevel as ByteLevelDecoder
from model import GPTConfig, GPT

#This is the location for the tokenizer and the losses and the checkpoint
tokenizer_location = '/content/drive/MyDrive/Software Engineering With Gen AI/Model/tokenizer.json'
losses_location = '/content/drive/MyDrive/Software Engineering With Gen AI/Model/losses3layer.csv'
checkpoint_location = '/content/drive/MyDrive/Software Engineering With Gen AI/Model/ckpt.pt'

#Gets the tokenizer
tokenizer = Tokenizer.from_file(
    tokenizer_location
)

device = "cuda" if torch.cuda.is_available() else "cpu"

#Gets the model
checkpoint = torch.load(
    checkpoint_location,
    map_location=device
)

config = GPTConfig(
    **checkpoint["model_args"]
)

model = GPT(config)

state_dict = checkpoint["model"]

# Handle nanoGPT checkpoint prefixes
for key in list(state_dict.keys()):
    if key.startswith("_orig_mod."):
        state_dict[key[len("_orig_mod."):]] = state_dict.pop(key)

model.load_state_dict(state_dict, strict=False)
model.to(device)
model.eval()

Mounted at /content/drive
number of parameters: 29.37M


GPT(
  (transformer): ModuleDict(
    (wte): Embedding(8192, 512)
    (wpe): Embedding(512, 512)
    (drop): Dropout(p=0.2, inplace=False)
    (h): ModuleList(
      (0-7): 8 x Block(
        (ln_1): LayerNorm()
        (attn): CausalSelfAttention(
          (c_attn): Linear(in_features=512, out_features=1536, bias=False)
          (c_proj): Linear(in_features=512, out_features=512, bias=False)
          (attn_dropout): Dropout(p=0.2, inplace=False)
          (resid_dropout): Dropout(p=0.2, inplace=False)
        )
        (ln_2): LayerNorm()
        (mlp): MLP(
          (c_fc): Linear(in_features=512, out_features=2048, bias=False)
          (gelu): GELU(approximate='none')
          (c_proj): Linear(in_features=2048, out_features=512, bias=False)
          (dropout): Dropout(p=0.2, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm()
  )
  (lm_head): Linear(in_features=512, out_features=8192, bias=False)
)

This function will parse the code and make sure that it is able to compile

In [3]:
import ast, re, time

def strip_extra_function(code):
  return code.split("\ndef ", 1)[0]


Loads the prompts

In [4]:
import random

random.seed(42)

prompts = [


    # Function name conflicts with instructions
    '''def square(x):
    """Return x cubed, not squared."""
    return ''',

    '''def add(a, b):
    """Subtract b from a instead of adding."""
    return ''',

    '''def reverse_list(items):
    """Return the original list without reversing it."""
    return ''',

    # Boundary requirements
    '''def is_positive(n):
    """Return True when n is zero or greater."""
    return n ''',

    # Different output requirements
    '''def find_max(numbers):
    """Return the smallest number in the list."""
    return ''',

    # Explicitly stated edge case
    '''def first_element(items):
    """Return None if the list is empty."""
    if not items:
        return ''',

     # 1. List comprehension
    """def filter_even(numbers):
    return [x for x in numbers if""",

    # 2. Dictionary comprehension
    """def create_dictionary(keys, values):
    return {k: v for k, v in""",

    # 3. Nested conditional
    """def classify_number(n):
    if n > 0:
        if n % 2 == 0:
            return""",

    # 4. Formatted string
    '''def greet(name):
    return f"Hello, {''',

    # 5. Nested loops
    """def print_pairs(items):
    for i in range(len(items)):
        for j in range(i + 1, len(items)):
            print(""",

    # 6. Try/except
    """def safe_divide(a, b):
    try:
        result = a / b
        return result
    except ZeroDivisionError:
        """,

    # 7. List indexing and slicing
    """def reverse_list(items):
    return items[""",

    # 8. Function with multiple conditions
    """def clamp(value, minimum, maximum):
    if value < minimum:
        return minimum
    elif value > maximum:
        return""",

    # 1. Bubble sort
    """def bubble_sort(arr):
    n = len(arr)
    for i in range(n):
        for j in range(0, n - i - 1):
            if arr[j] > arr[j + 1]:
                """,

    # 2. Binary search
    """def binary_search(arr, target):
    low = 0
    high = len(arr) - 1
    while low <= high:
        mid = (low + high) // 2
        """,

    # 3. Fibonacci
    """def fibonacci(n):
    if n <= 1:
        return n
    return """,

    # 4. Insertion sort
    """def insertion_sort(arr):
    for i in range(1, len(arr)):
        key = arr[i]
        j = i - 1
        """,

    # 5. Factorial
    """def factorial(n):
    if n <= 1:
        return 1
    return """,

    # 6. Linear search
    """def linear_search(arr, target):
    for i in range(len(arr)):
        if arr[i] == target:
            """,

    # 7. Euclidean algorithm (GCD)
    """def gcd(a, b):
    while b:
        a, b = b, """,

    # 8. Quick sort
    """def quick_sort(arr):
    if len(arr) <= 1:
        return arr
    pivot = arr[0]
    left = []
    right = []
    for x in arr[1:]:
        """,



    # 1. Find maximum - Variable updates
    """def find_max(numbers):
    largest = numbers[0]
    for number in numbers:
        if number > largest:
            """,

    # 2. Average - Arithmetic
    """def average(numbers):
    total = sum(numbers)
    return """,

    # 3. Count items - Loop logic
    """def count_items(items):
    count = 0
    for item in items:
        count """,

    # 4. Contains - Boolean logic
    """def contains(items, target):
    for item in items:
        if item == target:
            return True
    """,

    # 5. Reverse list - Indexing
    """def reverse_list(items):
    return """,

    # 6. Sum of digits - Loop updates
    """def sum_of_digits(n):
    total = 0
    while n > 0:
        total += """,

    # 7. Find minimum - Comparison logic
    """def find_min(numbers):
    smallest = numbers[0]
    for number in numbers:
        if number < smallest:
            """
]



for prompt in prompts:
    #Starts working code as False and checks it each time
    working_code = False
    ids = tokenizer.encode(prompt).ids

    x = torch.tensor(
        ids,
        dtype=torch.long,
        device=device
    )[None, :]


    with torch.no_grad():
        y = model.generate(
            x,
            max_new_tokens=50,
            temperature=0.3,
            top_k=3
        )

    output = tokenizer.decode(y[0].tolist())
    print(output)
    print('------------------------')
    output = strip_extra_function(output)
    print(output)
    print("=" * 60)

 def square(x):
    """Return x cubed, not squared."""
    return  # x^2 + y^2 + y^2 = x^2 - y^2
 
def main() -> None:
    x = [1, 2, 3, 4, 5]
    y = [
------------------------
 def square(x):
    """Return x cubed, not squared."""
    return  # x^2 + y^2 + y^2 = x^2 - y^2
 
 def add(a, b):
    """Subtract b from a instead of adding."""
    return  # type: ignore[attr-defined]
 
def __init__(self, size: int) -> None:
        self.size = size
        self.tree = [0] * size
 
def __
------------------------
 def add(a, b):
    """Subtract b from a instead of adding."""
    return  # type: ignore[attr-defined]
 
 def reverse_list(items):
    """Return the original list without reversing it."""
    return  # `items` (e.g. a `items` is `items` or `items`).
 
def _normalize_indexes(items: list[int], k: int) -> int:
    if k
------------------------
 def reverse_list(items):
    """Return the original list without reversing it."""
    return  # `items` (e.g. a `items` is `items` or `items`).

KeyboardInterrupt: 